In [1]:
import pandas as pd
import numpy as np

# Load clean data
ipl = pd.read_csv('E:\\ipl_dataset1\\ipl_clean.csv',
                   low_memory=False)

ipl['season'] = ipl['season'].astype(str).str[:4].astype(int)
ipl = ipl[ipl['season'] != 2026]
matches = ipl.drop_duplicates(subset='match_id')

print("Data loaded!")

Data loaded!


In [3]:
# Creating win_rate
wins = matches[matches['match_won_by'] != 'Unknown']\
       .groupby('match_won_by').size().reset_index(name='wins')

total = pd.concat([
    matches[['batting_team']].rename(columns={'batting_team':'team'}),
    matches[['bowling_team']].rename(columns={'bowling_team':'team'})
]).groupby('team').size().reset_index(name='total_matches')

win_rate = wins.merge(total, left_on='match_won_by', right_on='team')
win_rate['win_rate'] = (
    win_rate['wins'] / win_rate['total_matches'] * 100
).round(2)

# KPI values for power bi dashboard
print("Total Matches:", ipl['match_id'].nunique())
print("Total Runs:", ipl['runs_batter'].sum())
print("Total Seasons:", ipl['season'].nunique())
print("Highest Win Rate Team:", 
      win_rate.sort_values('win_rate', ascending=False).iloc[0]['team'],
      win_rate.sort_values('win_rate', ascending=False).iloc[0]['win_rate'], "%")

Total Matches: 1169
Total Runs: 355373
Total Seasons: 17
Highest Win Rate Team: Gujarat Titans 61.67 %


In [4]:
import os
os.makedirs('E:\\ipl_dataset1\\powerbi_data', exist_ok=True)

# Table 1 — Team Win Rate
wins = matches[matches['match_won_by'] != 'Unknown']\
       .groupby('match_won_by').size().reset_index(name='wins')

total = pd.concat([
    matches[['batting_team']].rename(
        columns={'batting_team':'team'}),
    matches[['bowling_team']].rename(
        columns={'bowling_team':'team'})
]).groupby('team').size().reset_index(name='total_matches')

win_rate = wins.merge(total, 
                      left_on='match_won_by', 
                      right_on='team')
win_rate['win_rate'] = (
    win_rate['wins'] / 
    win_rate['total_matches'] * 100
).round(2)
win_rate.to_csv('E:\\ipl_dataset1\\powerbi_data\\team_win_rate.csv', 
                index=False)
print(" Team win rate exported!")

# Table 2 — Season Runs Trend
season_runs = ipl.groupby('season').agg(
    total_runs=('runs_batter', 'sum'),
    total_matches=('match_id', 'nunique')
).reset_index()
season_runs['runs_per_match'] = (
    season_runs['total_runs'] / 
    season_runs['total_matches']
).round(2)
season_runs.to_csv('E:\\ipl_dataset1\\powerbi_data\\season_runs.csv',
                    index=False)
print(" Season runs exported!")

# Table 3 — Top Batsmen
top_batsmen = ipl.groupby('batter')['runs_batter']\
               .sum().reset_index(name='total_runs')\
               .sort_values('total_runs', ascending=False)\
               .head(10)
top_batsmen.to_csv('E:\\ipl_dataset1\\powerbi_data\\top_batsmen.csv',
                    index=False)
print(" Top batsmen exported!")

# Table 4 — Top Bowlers
wickets = ipl[~ipl['wicket_kind'].isin(
    ['none', 'run out', 'obstructing the field']
)]
top_bowlers = wickets.groupby('bowler')['wicket_kind']\
               .count().reset_index(name='total_wickets')\
               .sort_values('total_wickets', ascending=False)\
               .head(10)
top_bowlers.to_csv('E:\\ipl_dataset1\\powerbi_data\\top_bowlers.csv',
                    index=False)
print(" Top bowlers exported!")

# Table 5 — Venue Analysis
venue = matches.groupby('venue').size()\
        .reset_index(name='total_matches')\
        .sort_values('total_matches', ascending=False)\
        .head(10)
venue.to_csv('E:\\ipl_dataset1\\powerbi_data\\venue_analysis.csv',
              index=False)
print(" Venue analysis exported!")

# Table 6 — Phase Analysis
phase = ipl.groupby('phase').agg(
    total_runs=('runs_batter', 'sum'),
    total_balls=('runs_batter', 'count')
).reset_index()
phase['runs_per_ball'] = (
    phase['total_runs'] / phase['total_balls']
).round(3)
phase.to_csv('E:\\ipl_dataset1\\powerbi_data\\phase_analysis.csv',
              index=False)
print(" Phase analysis exported!")

print("\n All tables exported to powerbi_data folder!")

 Team win rate exported!
 Season runs exported!
 Top batsmen exported!
 Top bowlers exported!
 Venue analysis exported!
 Phase analysis exported!

 All tables exported to powerbi_data folder!
